In [ ]:
import os
import subprocess

print("Loading and verifying Palace from Google Drive...\n")

# 1. Mount Google Drive if not mounted
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")

# 2. Paths
DRIVE_PALACE_DIR = '/content/drive/MyDrive/Palace6'
BINARY_PATH = os.path.join(DRIVE_PALACE_DIR, 'build', 'bin', 'palace')
DRIVE_LIB_DIR = os.path.join(DRIVE_PALACE_DIR, 'build', 'lib')

print(f"Loading Palace binary from: {BINARY_PATH}")

# 3. Configure Runtime Environment
env = os.environ.copy()

# Add Drive dynamic libraries to LD_LIBRARY_PATH
current_ld = env.get('LD_LIBRARY_PATH', '')
env['LD_LIBRARY_PATH'] = f"{DRIVE_LIB_DIR}:{current_ld}" if current_ld else DRIVE_LIB_DIR
print(f"LD_LIBRARY_PATH set to: {env['LD_LIBRARY_PATH']}")

# Allow OpenMPI to execute under Colab's root user
env['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
env['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

# 4. Check & Execute
if os.path.exists(BINARY_PATH):
    os.chmod(BINARY_PATH, 0o755)
    print("Executable permissions verified.")

    print("\nTesting Palace execution...")
    try:
        # Try --version, fallback to --help if needed
        try:
            cmd = [BINARY_PATH, '--version']
            result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)
        except subprocess.CalledProcessError:
            cmd = [BINARY_PATH, '--help']
            result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)

        print("\n=========================================")
        print("✅ Palace successfully loaded and ran!")
        print(f"Command output ({' '.join(cmd)}):\n")
        print(result.stdout if result.stdout else result.stderr)
        print("=========================================")

    except subprocess.CalledProcessError as e:
        print("\n=========================================")
        print(f"❌ ERROR: Palace failed to run! Exit code: {e.returncode}")
        print(f"Stderr:\n{e.stderr}")
        print("=========================================")
else:
    print(f"\n❌ ERROR: Palace binary not found at: {BINARY_PATH}")

Loading and verifying Palace from Google Drive...

Google Drive already mounted.
Loading Palace binary from: /content/drive/MyDrive/Palace6/build/bin/palace
LD_LIBRARY_PATH set to: /content/drive/MyDrive/Palace6/build/lib
Executable permissions verified.

Testing Palace execution...

✅ Palace successfully loaded and ran!
Command output (/content/drive/MyDrive/Palace6/build/bin/palace --version):

>> /usr/bin/mpirun -n 1 /content/drive/MyDrive/Palace6/build/bin/palace-x86_64.bin --version

Palace version: v0.17.0-194-g01f5d9bd1
Schema version: 1-3-1



In [ ]:
import subprocess
import os

# Reuse the environment from the previous cell, which includes LD_LIBRARY_PATH
env = os.environ.copy()
DRIVE_PALACE_DIR = os.environ.get('DRIVE_PALACE_DIR', '/content/drive/MyDrive/Palace6')
BINARY_PATH = os.path.join(DRIVE_PALACE_DIR, 'build', 'bin', 'palace')
DRIVE_LIB_DIR = os.path.join(DRIVE_PALACE_DIR, 'build', 'lib')

current_ld = env.get('LD_LIBRARY_PATH', '')
if DRIVE_LIB_DIR not in current_ld:
    env['LD_LIBRARY_PATH'] = f"{DRIVE_LIB_DIR}:{current_ld}" if current_ld else DRIVE_LIB_DIR


print("Running Palace --help to verify execution...")
try:
    # Using --help as it was shown to work previously.
    cmd = [BINARY_PATH, '--help']
    result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)
    print("\n=========================================")
    print("✅ Palace --help executed successfully!")
    print(f"Output:\n{result.stdout}")
    print("=========================================")
except subprocess.CalledProcessError as e:
    print("\n=========================================")
    print(f"❌ ERROR: Palace --help failed! Exit code: {e.returncode}")
    print(f"Stderr:\n{e.stderr}")
    print("=========================================")
except FileNotFoundError:
    print(f"❌ ERROR: Palace binary not found at {BINARY_PATH}. Please ensure it is built and present.")


Running Palace --help to verify execution...

✅ Palace --help executed successfully!
Output:
Usage: palace [OPTIONS] CONFIG_FILE

Wrapper for launching Palace using MPI

Options:
  -h, --help                       Show this help message and exit
  -dry-run, --dry-run              Parse configuration file for errors and exit
  -serial, --serial                Call Palace without MPI launcher, default is false
  -np, --np NUM_PROCS              How many MPI processes to use, default is 1
  -nt, --nt NUM_THREADS            Number of OpenMP threads to use for OpenMP builds, default is 1 or the value of OMP_NUM_THREADS in the environment
  -launcher, --launcher LAUNCHER   MPI launcher, default is `mpirun`
  -launcher-args,
    --launcher-args ARGS           Any extra arguments to pass to MPI launcher, for example `--map-by` or `--bind-to` with their respective options (quoted)




In [ ]:
import os
import subprocess
import json

print("=========================================")
print("  PALACE INSTALLATION VERIFICATION CHECK ")
print("=========================================\n")

# 1. Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("✓ Google Drive is mounted.")

DRIVE_PALACE_DIR = '/content/drive/MyDrive/Palace6'
BUILD_DIR = os.path.join(DRIVE_PALACE_DIR, 'build')
BIN_DIR = os.path.join(BUILD_DIR, 'bin')
LIB_DIR = os.path.join(BUILD_DIR, 'lib')

# 2. Check Required Artifacts
required_files = [
    (os.path.join(BIN_DIR, 'palace'), "Palace Launcher Wrapper"),
    (os.path.join(BIN_DIR, 'palace-x86_64.bin'), "Palace C++ Solver Engine"),
    (os.path.join(BIN_DIR, 'validate-config'), "Config Validator Tool"),
    (os.path.join(BIN_DIR, 'palace_ldd.txt'), "Dependency LDD Log"),
    (os.path.join(BIN_DIR, 'palace_version.txt'), "Version/Help Output Log"),
    (os.path.join(BUILD_DIR, 'palace_commit.txt'), "Git Commit Metadata"),
    (os.path.join(BUILD_DIR, 'palace_git_status.txt'), "Git Status Log"),
    (os.path.join(BUILD_DIR, 'CMakeCache.txt'), "CMake Cache"),
    (os.path.join(BUILD_DIR, 'cmake_log.txt'), "CMake Build Log"),
    (os.path.join(BUILD_DIR, 'make_log.txt'), "Make Build Log"),
    (os.path.join(BUILD_DIR, 'build_tree.txt'), "Build Directory Tree Log"),
    (os.path.join(BUILD_DIR, 'use_palace.sh'), "Restore Launcher Script")
]

all_passed = True
print("\n--- 1. Checking Critical Files ---")
for file_path, label in required_files:
    if os.path.exists(file_path):
        print(f"  [PASS] Found {label}: {os.path.basename(file_path)}")
    else:
        print(f"  [FAIL] MISSING {label}: {file_path}")
        all_passed = False

# 3. Check Shared Libraries (.so)
print("\n--- 2. Checking Shared Libraries ---")
if os.path.isdir(LIB_DIR):
    so_files = [f for f in os.listdir(LIB_DIR) if '.so' in f]
    print(f"  [PASS] Found {len(so_files)} shared library files in {LIB_DIR}")
    for so in so_files[:5]:  # Show first few libraries
        print(f"         - {so}")
    if len(so_files) == 0:
        print("  [WARN] No .so files found in lib directory!")
else:
    print(f"  [FAIL] Library directory missing: {LIB_DIR}")
    all_passed = False

# 4. Check Executable Permissions
print("\n--- 3. Checking Executable Permissions ---")
binaries_to_check = [
    os.path.join(BIN_DIR, 'palace'),
    os.path.join(BIN_DIR, 'palace-x86_64.bin'),
    os.path.join(BUILD_DIR, 'use_palace.sh')
]

for b in binaries_to_check:
    if os.path.exists(b):
        os.chmod(b, 0o755)
        print(f"  [PASS] Verified executable permissions for {os.path.basename(b)}")

# 5. Execute Test Run (-dry-run)
# print("\n--- 4. Performing Execution Test ---")
# env = os.environ.copy()

# current_ld = env.get('LD_LIBRARY_PATH', '')
# env['LD_LIBRARY_PATH'] = f"{LIB_DIR}:{current_ld}" if current_ld else LIB_DIR
# env['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
# env['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

# config_file_path = os.path.join(DRIVE_PALACE_DIR, 'dummy_config.json')

# if not os.path.exists(config_file_path):
#     dummy_config = {
#         "Problem": {"Type": "Electrostatic", "Verbose": 1},
#         "Model": {"Mesh": "mesh.msh"},
#         "Domains": {"Postprocessing": {"Energy": []}},
#         "Boundaries": {"Ground": [1]},
#         "Solver": {"Linear": {"Type": "AMS", "KSPType": "CG", "Tol": 1e-6, "MaxIter": 100}}
#     }
#     with open(config_file_path, 'w') as f:
#         json.dump(dummy_config, f, indent=2)

# try:
#     cmd = [os.path.join(BIN_DIR, 'palace'), '-dry-run', config_file_path]
#     result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)
#     print("  [PASS] Palace -dry-run executed successfully!")
#     print("\nCommand Output:\n" + result.stdout)
# except subprocess.CalledProcessError as e:
#     print(f"  [FAIL] Palace -dry-run failed with exit code {e.returncode}")
#     print(f"Stderr:\n{e.stderr}")
#     all_passed = False
# except Exception as e:
#     print(f"  [FAIL] Execution error: {e}")
#     all_passed = False

# Final Summary
print("\n=========================================")
if all_passed:
    print("  SUCCESS: Palace installation is 100% complete and functional!")
else:
    print("  WARNING: One or more checks failed. Review output above.")
print("=========================================")

  PALACE INSTALLATION VERIFICATION CHECK 

✓ Google Drive is mounted.

--- 1. Checking Critical Files ---
  [PASS] Found Palace Launcher Wrapper: palace
  [PASS] Found Palace C++ Solver Engine: palace-x86_64.bin
  [PASS] Found Config Validator Tool: validate-config
  [PASS] Found Dependency LDD Log: palace_ldd.txt
  [PASS] Found Version/Help Output Log: palace_version.txt
  [PASS] Found Git Commit Metadata: palace_commit.txt
  [PASS] Found Git Status Log: palace_git_status.txt
  [PASS] Found CMake Cache: CMakeCache.txt
  [PASS] Found CMake Build Log: cmake_log.txt
  [PASS] Found Make Build Log: make_log.txt
  [PASS] Found Build Directory Tree Log: build_tree.txt
  [PASS] Found Restore Launcher Script: use_palace.sh

--- 2. Checking Shared Libraries ---
  [PASS] Found 7 shared library files in /content/drive/MyDrive/Palace6/build/lib
         - libxsmm.so.1.17.0
         - libceed.so
         - libxsmmgen.so.1.17.0
         - libxsmmgen.so.1
         - libxsmm.so.1

--- 3. Checking Exec

In [ ]:
import os

config_file_path = os.path.join(DRIVE_PALACE_DIR, 'dummy_config.json')
with open(config_file_path, 'w') as f:
    # Providing a more structured JSON to satisfy Palace's parsing requirements
    f.write('''
{
  "Problem": {
    "Type": "Electrostatic",
    "Solver": {
      "Type": "GMRES",
      "RelTol": 1.0e-6,
      "AbsTol": 1.0e-12,
      "MaxIter": 100
    }
  },
  "Domain": {},
  "Boundaries": {}
}
''')

print(f"Updated dummy configuration file: {config_file_path} with a more structured JSON.")


Updated dummy configuration file: /content/drive/MyDrive/Palace6/dummy_config.json with a more structured JSON.


In [ ]:
import os
import glob

build_dir = '/content/drive/MyDrive/Palace6/build'

print("Searching for actual compiled binaries inside build directory...\n")
for root, dirs, files in os.walk(build_dir):
    for file in files:
        full_path = os.path.join(root, file)
        # Find executable files that are NOT shell scripts
        if os.access(full_path, os.X_OK) and not file.endswith('.sh') and not file.endswith('.txt'):
            print(f"Found binary: {full_path}")

Searching for actual compiled binaries inside build directory...

Found binary: /content/drive/MyDrive/Palace6/build/bin/palace
Found binary: /content/drive/MyDrive/Palace6/build/bin/palace-x86_64.bin
Found binary: /content/drive/MyDrive/Palace6/build/bin/validate-config
Found binary: /content/drive/MyDrive/Palace6/build/bin/schema/config-schema.json
Found binary: /content/drive/MyDrive/Palace6/build/bin/schema/ValidateConfig.jl
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmm.so.1.17.0
Found binary: /content/drive/MyDrive/Palace6/build/lib/libceed.so
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmmgen.so.1.17.0
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmmgen.so.1
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmm.so.1
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmm.so
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmmgen.so


In [ ]:
import os

LIB_DIR = '/content/drive/MyDrive/Palace6/build/lib'

# Libraries that require symlinks for runtime binding
symlinks_to_create = [
    ('libxsmm.so.1.17.0', 'libxsmm.so.1'),
    ('libxsmm.so.1.17.0', 'libxsmm.so'),
    ('libxsmmgen.so.1.17.0', 'libxsmmgen.so.1'),
    ('libxsmmgen.so.1.17.0', 'libxsmmgen.so')
]

print("Creating missing library symlinks in Google Drive...\n")

for target, link_name in symlinks_to_create:
    target_path = os.path.join(LIB_DIR, target)
    link_path = os.path.join(LIB_DIR, link_name)

    if os.path.exists(target_path):
        if os.path.islink(link_path) or os.path.exists(link_path):
            os.remove(link_path)
        os.symlink(target, link_path)
        print(f"  [CREATED] {link_name} -> {target}")
    else:
        print(f"  [SKIP] Target file {target} not found in {LIB_DIR}")

print("\n✅ Symlinks created successfully!")

Creating missing library symlinks in Google Drive...

  [CREATED] libxsmm.so.1 -> libxsmm.so.1.17.0
  [CREATED] libxsmm.so -> libxsmm.so.1.17.0
  [CREATED] libxsmmgen.so.1 -> libxsmmgen.so.1.17.0
  [CREATED] libxsmmgen.so -> libxsmmgen.so.1.17.0

✅ Symlinks created successfully!


In [ ]:
import os
import subprocess
import json

print("=========================================")
print("  PALACE INSTALLATION VERIFICATION CHECK ")
print("=========================================\n")

# 1. Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("✓ Google Drive is mounted.")

DRIVE_PALACE_DIR = '/content/drive/MyDrive/Palace6'
BUILD_DIR = os.path.join(DRIVE_PALACE_DIR, 'build')
BIN_DIR = os.path.join(BUILD_DIR, 'bin')
LIB_DIR = os.path.join(BUILD_DIR, 'lib')

# 2. Check Required Artifacts
required_files = [
    (os.path.join(BIN_DIR, 'palace'), "Palace Launcher Wrapper"),
    (os.path.join(BIN_DIR, 'palace-x86_64.bin'), "Palace C++ Solver Engine"),
    (os.path.join(BIN_DIR, 'validate-config'), "Config Validator Tool"),
    (os.path.join(BIN_DIR, 'palace_ldd.txt'), "Dependency LDD Log"),
    (os.path.join(BIN_DIR, 'palace_version.txt'), "Version/Help Output Log"),
    (os.path.join(BUILD_DIR, 'palace_commit.txt'), "Git Commit Metadata"),
    (os.path.join(BUILD_DIR, 'palace_git_status.txt'), "Git Status Log"),
    (os.path.join(BUILD_DIR, 'CMakeCache.txt'), "CMake Cache"),
    (os.path.join(BUILD_DIR, 'cmake_log.txt'), "CMake Build Log"),
    (os.path.join(BUILD_DIR, 'make_log.txt'), "Make Build Log"),
    (os.path.join(BUILD_DIR, 'build_tree.txt'), "Build Directory Tree Log"),
    (os.path.join(BUILD_DIR, 'use_palace.sh'), "Restore Launcher Script")
]

all_passed = True
print("\n--- 1. Checking Critical Files ---")
for file_path, label in required_files:
    if os.path.exists(file_path):
        print(f"  [PASS] Found {label}: {os.path.basename(file_path)}")
    else:
        print(f"  [FAIL] MISSING {label}: {file_path}")
        all_passed = False

# 3. Check Shared Libraries (.so)
print("\n--- 2. Checking Shared Libraries ---")
if os.path.isdir(LIB_DIR):
    so_files = [f for f in os.listdir(LIB_DIR) if '.so' in f]
    print(f"  [PASS] Found {len(so_files)} shared library files in {LIB_DIR}")
    for so in so_files[:5]:  # Show first few libraries
        print(f"         - {so}")
    if len(so_files) == 0:
        print("  [WARN] No .so files found in lib directory!")
else:
    print(f"  [FAIL] Library directory missing: {LIB_DIR}")
    all_passed = False

# 4. Check Executable Permissions
print("\n--- 3. Checking Executable Permissions ---")
binaries_to_check = [
    os.path.join(BIN_DIR, 'palace'),
    os.path.join(BIN_DIR, 'palace-x86_64.bin'),
    os.path.join(BUILD_DIR, 'use_palace.sh')
]

for b in binaries_to_check:
    if os.path.exists(b):
        os.chmod(b, 0o755)
        print(f"  [PASS] Verified executable permissions for {os.path.basename(b)}")

# # 5. Execute Test Run (-dry-run)
# print("\n--- 4. Performing Execution Test ---")
# env = os.environ.copy()

# current_ld = env.get('LD_LIBRARY_PATH', '')
# env['LD_LIBRARY_PATH'] = f"{LIB_DIR}:{current_ld}" if current_ld else LIB_DIR
# env['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
# env['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

# config_file_path = os.path.join(DRIVE_PALACE_DIR, 'dummy_config.json')

# if not os.path.exists(config_file_path):
#     dummy_config = {
#         "Problem": {"Type": "Electrostatic", "Verbose": 1},
#         "Model": {"Mesh": "mesh.msh"},
#         "Domains": {"Postprocessing": {"Energy": []}},
#         "Boundaries": {"Ground": [1]},
#         "Solver": {"Linear": {"Type": "AMS", "KSPType": "CG", "Tol": 1e-6, "MaxIter": 100}}
#     }
#     with open(config_file_path, 'w') as f:
#         json.dump(dummy_config, f, indent=2)

# try:
#     cmd = [os.path.join(BIN_DIR, 'palace'), '-dry-run', config_file_path]
#     result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)
#     print("  [PASS] Palace -dry-run executed successfully!")
#     print("\nCommand Output:\n" + result.stdout)
# except subprocess.CalledProcessError as e:
#     print(f"  [FAIL] Palace -dry-run failed with exit code {e.returncode}")
#     print(f"Stderr:\n{e.stderr}")
#     all_passed = False
# except Exception as e:
#     print(f"  [FAIL] Execution error: {e}")
#     all_passed = False

# Final Summary
print("\n=========================================")
if all_passed:
    print("  SUCCESS: Palace installation is 100% complete and functional!")
else:
    print("  WARNING: One or more checks failed. Review output above.")
print("=========================================")

  PALACE INSTALLATION VERIFICATION CHECK 

✓ Google Drive is mounted.

--- 1. Checking Critical Files ---
  [PASS] Found Palace Launcher Wrapper: palace
  [PASS] Found Palace C++ Solver Engine: palace-x86_64.bin
  [PASS] Found Config Validator Tool: validate-config
  [PASS] Found Dependency LDD Log: palace_ldd.txt
  [PASS] Found Version/Help Output Log: palace_version.txt
  [PASS] Found Git Commit Metadata: palace_commit.txt
  [PASS] Found Git Status Log: palace_git_status.txt
  [PASS] Found CMake Cache: CMakeCache.txt
  [PASS] Found CMake Build Log: cmake_log.txt
  [PASS] Found Make Build Log: make_log.txt
  [PASS] Found Build Directory Tree Log: build_tree.txt
  [PASS] Found Restore Launcher Script: use_palace.sh

--- 2. Checking Shared Libraries ---
  [PASS] Found 7 shared library files in /content/drive/MyDrive/Palace6/build/lib
         - libxsmm.so.1.17.0
         - libceed.so
         - libxsmmgen.so.1.17.0
         - libxsmmgen.so.1
         - libxsmm.so.1

--- 3. Checking Exec